In [1]:
import torch
from torch_geometric.nn import knn_graph, knn


import numpy as np
from tqdm import tqdm

import os
import glob
import re

# sys.path.append('../Utility Scripts')

from Utility_functions import print_3D_graph

import params

print('NSIM:', params.NSIM)
print('NSTEP:', params.NSTEP)
print('FLUID_PROBES:', params.NFLUID_PROBES)
print('IO_PROBES:', params.NIO_PROBES)
print('NPOS:', params.NPOS)
print('NFEATURES:', params.NFEATURES)
print('DATADIR:', params.DATADIR)

NSIM: 10
NSTEP: 7000
FLUID_PROBES: 61
IO_PROBES: 1
NPOS: 6
NFEATURES: 10
DATADIR: ../.data/Dataset_10sims_5G2N


In [2]:

#dati = np.load("../.data/Extracted_data/probes/SIM000_probepos.npy")
dati = np.load("../.data/Dataset_10sims_10G2N/probes/SIM000_probepos.npy")

print(len(dati))
print(dati) 

2046
[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  4.00000000e-03
   4.00000000e-03  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00 -1.00000000e-01  3.60000000e-03
   3.60000000e-03  2.00000000e+00]
 [-9.40868136e-04  2.26281525e-02 -1.22249007e-01  3.20000000e-03
   3.20000000e-03  2.00000000e+00]
 ...
 [ 2.73825017e-02 -1.85197993e-02 -2.62808135e-01  6.03979776e-05
   6.03979776e-05  2.00000000e+00]
 [ 2.82341420e-02 -1.92422083e-02 -2.62169323e-01  6.76414963e-05
   6.76414963e-05  2.00000000e+00]
 [ 2.83991217e-02 -1.94385136e-02 -2.62968030e-01  6.03979776e-05
   6.03979776e-05  1.00000000e+00]]


In [3]:

data = np.load("../.data/Dataset_10sims_10G2N/fields/SIM000_features.npz")
#data = np.load("../.data/Dataset_10sims_Moebius/fields/SIM000_probes.npz")


print(data.files)

for name in data.files:
    print(f"{name}:")
    print(data[name].tolist())
    print(data[name].shape)
    print(len(data[name]))
    print("\n" + "-"*50 + "\n")  

['edges_pressure_array', 'edges_velocity_array_x', 'edges_velocity_array_y', 'edges_velocity_array_z', 'stress_xx', 'stress_yy', 'stress_zz', 'stress_xy', 'stress_xz', 'stress_yz']
edges_pressure_array:
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0

# READING NPZ AND NPY

In [4]:
numbers = re.compile(r'(\d+)')

def numericalSort(value):
    parts = numbers.split(value)
    parts[1::2] = map(int, parts[1::2])
    return parts

In [5]:
probes_path = os.path.join(params.DATADIR, 'probes')
probes = sorted(glob.glob(os.path.join(probes_path, '*.npy')), key = numericalSort)
print(probes[0])

fields_path = os.path.join(params.DATADIR, 'fields')
fields = sorted(glob.glob(os.path.join(fields_path, '*.npz')), key = numericalSort)
print(fields[0])

../.data/Dataset_10sims_5G2N\probes\SIM000_probepos.npy
../.data/Dataset_10sims_5G2N\fields\SIM000_features.npz


In [6]:
# probe_data_list = []

# for data_file in probes:
#     read_points = np.load(data_file)
#     probe_data_list.append(torch.from_numpy(read_points))

# pos_list = probe_data_list
# #print(pos_list)
# print(len(pos_list))
# print(pos_list[0].tolist())
# print(len(pos_list[0]))
# print(pos_list[0])


In [7]:
# features_list = []

# for data_file in fields:
#     read_features = np.load(data_file)
#     num_points = None
#     for _, filedat in enumerate(read_features.files):
#         if num_points is None:
#             num_points = len(read_features[filedat])
#             print(len(read_features[filedat]))
#             feature_tensor = torch.zeros(num_points, 10)
#             print(len(feature_tensor))
#         #print(read_features[filedat])
#         feature_tensor[:, _] = torch.from_numpy(read_features[filedat]).squeeze()
#     features_list.append(feature_tensor)

# feat_list = features_list
# # print(feat_list)
# # print(len(feat_list))
# # print(pos_list[0].tolist())
# # print(len(feat_list[0]))
# # print(feat_list[0])

In [8]:
print(len(probes))
assert params.NSIM == len(probes)

dataset_len = len(probes)  # number of simulations
print(probes)
positions = np.empty((params.NSIM, params.NPOS, params.NFLUID_PROBES + params.NIO_PROBES))
#print(positions)
for i, data in enumerate(probes):
    read_points = np.load(data)
    #print(read_points)
    positions[i,:] = read_points.T
    
pos = torch.from_numpy(np.einsum('ijk->ikj', positions))
print(len(pos))
print(pos[0])
#print(pos[1])
print(pos.shape) # [params.NSIM, params.NFLUID_PROBES + params.NIO_PROBES, params.NPOS]

10
['../.data/Dataset_10sims_5G2N\\probes\\SIM000_probepos.npy', '../.data/Dataset_10sims_5G2N\\probes\\SIM001_probepos.npy', '../.data/Dataset_10sims_5G2N\\probes\\SIM002_probepos.npy', '../.data/Dataset_10sims_5G2N\\probes\\SIM003_probepos.npy', '../.data/Dataset_10sims_5G2N\\probes\\SIM004_probepos.npy', '../.data/Dataset_10sims_5G2N\\probes\\SIM005_probepos.npy', '../.data/Dataset_10sims_5G2N\\probes\\SIM006_probepos.npy', '../.data/Dataset_10sims_5G2N\\probes\\SIM007_probepos.npy', '../.data/Dataset_10sims_5G2N\\probes\\SIM008_probepos.npy', '../.data/Dataset_10sims_5G2N\\probes\\SIM009_probepos.npy']
10
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  4.0000e-03,  4.0000e-03,
          0.0000e+00],
        [ 0.0000e+00,  0.0000e+00, -1.0000e-01,  3.6000e-03,  3.6000e-03,
          2.0000e+00],
        [ 1.6149e-03,  1.8724e-02, -1.2559e-01,  3.2000e-03,  3.2000e-03,
          2.0000e+00],
        [ 3.2297e-03,  3.7447e-02, -1.5118e-01,  2.2859e-03,  2.2859e-03,
          2.0000e+

In [9]:
# features = np.empty((params.NSIM, params.NSIM * NUM_FEATURES, NUM_NODES))
####positions = np.empty((params.NSIM, params.NPOS, params.NFLUID_PROBES + params.NIO_PROBES))
features = np.zeros((params.NSIM, params.NFRAME * params.NFEATURES, params.NFLUID_PROBES + params.NIO_PROBES))
feat = features.reshape((params.NSIM, params.NFRAME, params.NFEATURES, params.NFLUID_PROBES + params.NIO_PROBES))
print(features.shape)
#print(feat)
print('Features shape:', features.shape)     # [1000, 60 (6*10), 2000]
#print(fields)
for i, data in enumerate(tqdm(fields)):

    read_features = np.load(data)
    #print(read_features.files)
    #print(read_features)
    for j, filedat in enumerate(read_features.files): 
        print(i, j, filedat, read_features[filedat].shape, features[i, j, :].shape)
        #features[i, j, :params.NFLUID_PROBES] = read_features[filedat]
        features[i, j, :] = read_features[filedat]
        #print(features[i, j, :params.NFLUID_PROBES])

print(features[0][0].tolist())

(10, 10, 62)
Features shape: (10, 10, 62)


  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 138.15it/s]

0 0 edges_pressure_array (62,) (62,)
0 1 nodes_velocity_array_x (62,) (62,)
0 2 nodes_velocity_array_y (62,) (62,)
0 3 nodes_velocity_array_z (62,) (62,)
0 4 stress_xx (62,) (62,)
0 5 stress_yy (62,) (62,)
0 6 stress_zz (62,) (62,)
0 7 stress_xy (62,) (62,)
0 8 stress_xz (62,) (62,)
0 9 stress_yz (62,) (62,)
1 0 edges_pressure_array (62,) (62,)
1 1 nodes_velocity_array_x (62,) (62,)
1 2 nodes_velocity_array_y (62,) (62,)
1 3 nodes_velocity_array_z (62,) (62,)
1 4 stress_xx (62,) (62,)
1 5 stress_yy (62,) (62,)
1 6 stress_zz (62,) (62,)
1 7 stress_xy (62,) (62,)
1 8 stress_xz (62,) (62,)
1 9 stress_yz (62,) (62,)
2 0 edges_pressure_array (62,) (62,)
2 1 nodes_velocity_array_x (62,) (62,)
2 2 nodes_velocity_array_y (62,) (62,)
2 3 nodes_velocity_array_z (62,) (62,)
2 4 stress_xx (62,) (62,)
2 5 stress_yy (62,) (62,)
2 6 stress_zz (62,) (62,)
2 7 stress_xy (62,) (62,)
2 8 stress_xz (62,) (62,)
2 9 stress_yz (62,) (62,)
3 0 edges_pressure_array (62,) (62,)
3 1 nodes_velocity_array_x (62,) 

In [10]:
#print(features[0][1].tolist()) # vel_x
feat = features.reshape((params.NSIM, params.NFRAME, params.NFEATURES, params.NFLUID_PROBES + params.NIO_PROBES))
print(feat.shape)
print(feat[0][0]) # vel_x
feat = torch.from_numpy(np.einsum('ijkl->ijlk', feat))
print(feat.shape)
print(feat[0][0][0]) # vel_x
#print(len(feat[0][0]))

(10, 1, 10, 62)
[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.0000000

In [11]:
# new_pos_list = [pos.unsqueeze(0).repeat(params.NFRAME, 1, 1) for pos in pos_list]
# print('Node position and labels data:', [pos.shape for pos in new_pos_list])
# print('Fluid dynamic features   data:', [feat.shape for feat in feat_list])

# dataset = [torch.cat([new_pos, feat.unsqueeze(0)], dim=-1) for new_pos, feat in zip(new_pos_list, feat_list)]

# print('Concatenated total data:', [data.shape for data in dataset])


In [12]:
new_pos = pos.unsqueeze(1).repeat(1, params.NFRAME, 1, 1)
print('Node position and labels data:', new_pos.shape)
print('Fluid dynamic features   data:', feat.shape)

dataset = torch.cat([new_pos, feat], dim = -1)
print(len(dataset))
print(dataset[0][0][13])

print('Concatenated total data:', dataset.shape) # [100,6,2100,16]

Node position and labels data: torch.Size([10, 1, 62, 6])
Fluid dynamic features   data: torch.Size([10, 1, 62, 10])
10
tensor([ 1.4389e-02, -3.4368e-02, -1.9514e-01,  1.4514e-03,  1.4514e-03,
         2.0000e+00,  0.0000e+00,  1.1846e-01, -9.5545e-02, -2.9720e-01,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00], dtype=torch.float64)
Concatenated total data: torch.Size([10, 1, 62, 16])


In [13]:
# cut to last frame, we are interested in the steady state

print('Data with all time frames:', dataset.shape)     # [1000,6,2100,16] [samples,frames, nodes, tot_features]

new_dataset = dataset[:, -1:, :, :].squeeze(1) # only last frame
print(new_dataset[0][13]) # coord_x, coord_y, coord_z, mis, size, sdf, press, vel_x, vel_y, vel_z, stress_ij

print('Data with only last frame:', new_dataset.shape)    # [samples, 2100, 16]

Data with all time frames: torch.Size([10, 1, 62, 16])
tensor([ 1.4389e-02, -3.4368e-02, -1.9514e-01,  1.4514e-03,  1.4514e-03,
         2.0000e+00,  0.0000e+00,  1.1846e-01, -9.5545e-02, -2.9720e-01,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00], dtype=torch.float64)
Data with only last frame: torch.Size([10, 62, 16])


# PREPROCESSING OF MIS VALUES

In [14]:
# clean up MIS to have smooth values next to wall (due to the staircase-like discretization)

spacing = params.SPACING
kneigh_shells = params.NEIGHBOURS

idx = 1
nodes = new_dataset.shape[1]
edge_index = knn_graph(new_dataset[idx,:,:3], 10)

print_3D_graph(new_dataset[idx,:,:3], edge_index, color = new_dataset[idx,:,4]) # original

for data in tqdm(new_dataset):

    # selecting near-surface nodes (outer shell)
    idx_shell1 = (data[:,3] <= spacing)
    pos_shell1 = data[idx_shell1,:3]

    # selecting near-to-outer shell nodes
    idx_shell2 = (data[:,3]>spacing)*(data[:,3]<=3*spacing)
    pos_shell2 = data[idx_shell2,:3]

    # ordered array of the two shells' nodes
    shells_pos = torch.cat((pos_shell1,pos_shell2),0)

    # creating edges between set1 and set2
    edge_indexes = knn(pos_shell2,pos_shell1, kneigh_shells)

    # storing indexes of edge-receiving nodes
    edge_reshaped = edge_indexes.reshape(-1)[int(sum(idx_shell1)*kneigh_shells):]

    # get the MIS value for the edge-receiving nodes
    sdf_reshaped = data[idx_shell2][edge_reshaped][:,4].reshape(-1,kneigh_shells)

    def most_frequent(row):
        i = 0
        values, counts = np.unique(row, return_counts=True)
        ind = np.argmax(counts)
        return values[ind]

    # for every row, calculate the most occurring MIS value
    # the i-th row represents the i-th edge-sending node (nodes belonging to the outer shell)
    new_val = np.apply_along_axis(most_frequent, 1, sdf_reshaped)

    # change outer shell's MIS values with the ones computed now
    data[idx_shell1,4] = torch.tensor(new_val)

    # indexing on the whole set of points has to be coherent with nodes ordering 
    # (knn function with two sets of nodes, orders both indexing from 0)
    edge_indexes_plot = edge_indexes.clone()
    edge_indexes_plot[1,:] += edge_indexes_plot[0,-1]+1


print_3D_graph(new_dataset[idx,:,:3], edge_index, color = new_dataset[idx,:,4]) # processed

100%|██████████| 10/10 [00:00<00:00, 401.66it/s]


# READ A GRAPH AND CONSTRUCT EDGE CONNECTIVITY

In [15]:
# check if given the big tensor storing all simulations, I am able to recover a specific steady state and its quantities

single_graph = new_dataset[0,...]

i=0
while i < 61:#len(single_graph):
    print(f"{i}th node: {torch.norm(single_graph[i,7:10])}")
    i += 1

neighbours = params.NEIGHBOURS

edge_index = knn_graph(single_graph[..., :3], neighbours)
print(single_graph[1]) # coord_x, coord_y, coord_z, mis, sdf, size, press, vel_x, vel_y, vel_z, stress_ij
# print(single_graph[1])
# print(single_graph[2])
# print(single_graph[3])
# print(single_graph[4])
#magvel = torch.norm(single_graph[..., 4:7], dim=-1)
magvel = torch.norm(single_graph[..., 7:10], dim=-1) # 7:10 are vel_x, vel_y, vel_z
#print(magvel)   

print('Single graph:', single_graph.shape)
print('Edge index  :', edge_index.shape)
print('Velocity magnitude:', magvel.shape)

print_3D_graph(single_graph, edge_index, magvel)

# [(0, -1), (0, 1), (1, 2), (1, 4), (2, 3),
# (4, 5), (3, 6), (3, 8), (5, 10), (5, 12), 
# (6, 7), (8, 9), (10, 11), (12, 13)]

0th node: 1.0
1th node: 1.0
2th node: 0.42728189742805867
3th node: 0.42728189742805867
4th node: 0.5727181025719416
5th node: 0.5727181025719416
6th node: 0.15198042350392038
7th node: 0.15198042350392038
8th node: 0.2753014739241384
9th node: 0.2753014739241384
10th node: 0.23882008274546784
11th node: 0.23882008274546784
12th node: 0.33389801982647377
13th node: 0.33389801982647377
14th node: 0.06761087027687956
15th node: 0.06761087027687956
16th node: 0.08436955322704087
17th node: 0.08436955322704087
18th node: 0.08923268328077726
19th node: 0.08923268328077726
20th node: 0.1860687906433613
21th node: 0.1860687906433613
22th node: 0.09801850938673293
23th node: 0.09801850938673293
24th node: 0.14080157335873508
25th node: 0.14080157335873508
26th node: 0.13090055230842365
27th node: 0.13090055230842365
28th node: 0.20299746751805048
29th node: 0.20299746751805048
30th node: 0.028262008710251106
31th node: 0.028262008710251106
32th node: 0.03934886156662843
33th node: 0.0393488615

# CHECKING MIS PREPROCESSING

In [16]:
idx = 0 # steady state idx
neighbours = params.NEIGHBOURS
nodes = new_dataset.shape[1]
new_dataset_copy = new_dataset.clone()

# print(new_dataset_copy.shape)

edge_index = knn_graph(new_dataset_copy[idx,:,:3], neighbours)
print('knn_graph:', edge_index.shape)

print_3D_graph(new_dataset_copy[idx,:,:3], edge_index, color = new_dataset_copy[idx,:,4]) # original    
print_3D_graph(new_dataset[idx,:,:3], edge_index, color = new_dataset[idx,:,4]) # processed

knn_graph: torch.Size([2, 186])


In [17]:
counts = []

# checking number of neighbors per node
for i in range(params.NFLUID_PROBES + params.NIO_PROBES):
    count = (edge_index[1,:] == i)
    count = count.sum()
    counts.append(count)

print('edge_index[:10]', edge_index[:10])
print('counts[:10]:', counts[:10])

count = torch.tensor(counts, dtype=torch.float32)
print('Mean:', count.mean(), 'Std:', count.std())

edge_index[:10] tensor([[ 1,  2,  4,  2,  4,  5,  4,  1,  3,  6,  8,  2,  2,  5,  1, 12, 10,  4,
          3,  7,  8, 16, 14,  6,  3,  9,  6, 18, 20,  8, 12,  5, 11, 22, 24, 10,
         10,  5, 13, 28, 26, 12, 15,  7, 16, 30, 32, 14, 17,  7, 14, 34, 36, 16,
         19,  9, 20, 40, 38, 18,  9, 21, 18, 44, 42, 20, 24, 23, 11, 46, 48, 22,
         22, 25, 11, 50, 52, 24, 28, 13, 27, 54, 56, 26, 26, 13, 29, 58, 60, 28,
         31, 15, 32, 30, 32, 15, 15, 33, 30, 32, 30, 15, 36, 17, 35, 34, 36, 37,
         34, 37, 17, 36, 34, 35, 39, 19, 40, 38, 40, 19, 19, 41, 38, 40, 38, 19,
         21, 43, 44, 42, 44, 21, 21, 45, 42, 44, 42, 21, 23, 47, 48, 46, 48, 23,
         49, 23, 46, 48, 46, 23, 25, 51, 52, 50, 52, 25, 53, 25, 50, 52, 50, 25,
         27, 55, 56, 54, 56, 27, 27, 57, 54, 56, 54, 27, 59, 29, 60, 58, 60, 29,
         61, 29, 58, 60, 58, 29],
        [ 0,  0,  0,  1,  1,  1,  2,  2,  2,  3,  3,  3,  4,  4,  4,  5,  5,  5,
          6,  6,  6,  7,  7,  7,  8,  8,  8,  9,  9,  9, 10

# SAVING DATASET

In [18]:
saved_dataset = new_dataset.reshape(-1, params.NFLUID_PROBES + params.NIO_PROBES, params.NFEATURES + 6).float()


In [19]:
raw_path = os.path.join(params.DATADIR, 'raw')
processed_path = os.path.join(params.DATADIR, 'processed')
dataset_name = 'Dataset.pt'

if not os.path.exists(raw_path):
    os.makedirs(raw_path)

if not os.path.exists(processed_path):
    os.makedirs(processed_path)

torch.save(saved_dataset, os.path.join(raw_path, dataset_name))